# 실습 3: 협업 필터링 Apply (EDA + Item-based CF 구현)

이번 실습은 두 파트로 나뉩니다.
- **Part 1**: MovieLens 데이터를 탐색(EDA)하며 평점 분포와 희소성(Sparsity)을 직접 눈으로 확인합니다.
- **Part 2**: `lab_01`의 User-based CF 대신 Item-based CF를 직접 구현하며 두 방식의 차이를 체감합니다.

`lab_02`에서 배운 원리를 바탕으로, 각 코드가 왜 그렇게 작성되었는지 생각하며 진행하세요.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Team-AnI/A-AND-I-4TH-AI-CODE-LAB/blob/main/4주차/lab_03_apply.ipynb)

> 위 배지를 누르면 이 노트북이 **여러분 Google 계정의 Colab**에서 열립니다. 수정본을 남기려면 `파일 → Drive에 사본 저장`.

## 1. 환경 설정

In [ ]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-100k.zip -O ml-100k.zip
!unzip -q -o ml-100k.zip

import pandas as pd
import torch
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

df = pd.read_csv(
    'ml-100k/u.data', sep='\t', header=None,
    names=['user_id', 'item_id', 'rating', 'timestamp']
)
movies = pd.read_csv(
    'ml-100k/u.item', sep='|', header=None, encoding='latin-1',
    usecols=[0, 1], names=['item_id', 'title']
)
movie_names = dict(zip(movies['item_id'], movies['title']))

n_users = df['user_id'].max()
n_items = df['item_id'].max()

ratings = torch.zeros(n_users, n_items)
for row in df.itertuples():
    ratings[row.user_id - 1, row.item_id - 1] = row.rating

print(f"데이터 로드 완료: {n_users}명 사용자 × {n_items}편 영화")

---
# Part 1: EDA (탐색적 데이터 분석)

모델을 만들기 전에 데이터를 먼저 파악합니다. '데이터가 어떻게 생겼는가'를 아는 것이 좋은 추천 시스템의 출발점입니다.

## 2-1. 평점 분포 확인
사용자들이 어떤 점수를 주로 주는지 확인합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 평점 값 분포
rating_counts = df['rating'].value_counts().sort_index()
axes[0].bar(rating_counts.index, rating_counts.values, color='steelblue', edgecolor='white')
axes[0].set_title('Rating Value Distribution')
axes[0].set_xlabel('Rating (1~5)')
axes[0].set_ylabel('Count')
axes[0].set_xticks([1, 2, 3, 4, 5])

# 평점 누적 분포
axes[1].pie(
    rating_counts.values,
    labels=[f'{r}점' for r in rating_counts.index],
    autopct='%1.1f%%',
    colors=plt.cm.Blues(np.linspace(0.3, 0.9, 5))
)
axes[1].set_title('Rating Distribution (%)')

plt.tight_layout()
plt.show()

print(f"평균 평점: {df['rating'].mean():.2f}")
print(f"중앙값 평점: {df['rating'].median():.1f}")

### 🔬 코드 해설
- 평점 분포를 보면 대부분의 사람이 3~4점을 많이 준다는 것을 알 수 있습니다. 이는 **긍정 편향(Positive Bias)** 이라고 합니다. 영화를 보러 가는 사람들은 이미 어느 정도 볼 가능성이 있는 영화를 선택하기 때문에 낮은 평점이 상대적으로 적습니다.
- 이러한 분포 특성은 추천 시스템 설계 시 고려해야 할 중요한 데이터 특성입니다.

## 2-2. 희소성(Sparsity) 시각화
행렬이 얼마나 비어 있는지를 수치와 시각적으로 확인합니다.

In [ ]:
total_cells = n_users * n_items
rated_cells = len(df)
sparsity = 1 - rated_cells / total_cells

print(f"전체 칸 수: {total_cells:,}  ({n_users} × {n_items})")
print(f"평점 있는 칸: {rated_cells:,}")
print(f"희소도(Sparsity): {sparsity:.2%}  → 전체의 {sparsity:.2%}가 비어 있음")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 행렬 일부 시각화 (첫 50 사용자 × 100 영화)
subset = ratings[:50, :100].numpy()
axes[0].imshow(subset, aspect='auto', cmap='YlOrRd', interpolation='nearest')
axes[0].set_title('Rating Matrix Heatmap\n(first 50 users × 100 movies)')
axes[0].set_xlabel('Movie ID')
axes[0].set_ylabel('User ID')
plt.colorbar(axes[0].images[0], ax=axes[0], label='Rating')

# 오른쪽: Sparsity 비율 막대
axes[1].bar(['Rated', 'Unrated'], [rated_cells, total_cells - rated_cells],
            color=['steelblue', 'lightgray'], edgecolor='white')
axes[1].set_title(f'Rated vs Unrated Cells\n(Sparsity: {sparsity:.2%})')
axes[1].set_ylabel('Number of Cells')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

plt.tight_layout()
plt.show()

### 🔬 코드 해설
- 희소도가 **93% 이상**이라는 것은, 한 사용자가 전체 영화의 7% 미만만 봤다는 뜻입니다. 이 상황에서 두 사용자가 공통으로 평가한 영화가 거의 없다면, 코사인 유사도 계산 자체가 의미 없어집니다.
- 히트맵에서 흰색(0) 칸이 절대다수를 차지하는 것이 보입니다. 이 구조가 협업 필터링의 Sparsity 문제를 직관적으로 보여줍니다.

## 2-3. 사용자/영화별 평점 수 분포
모든 사용자와 영화가 균등하게 평가되는 것이 아닙니다. 인기 있는 영화와 활발한 사용자에 데이터가 집중되는 '롱테일(Long Tail)' 현상을 확인합니다.

In [ ]:
user_rating_counts = df.groupby('user_id').size()
item_rating_counts = df.groupby('item_id').size()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 사용자별 평점 수 분포
axes[0, 0].hist(user_rating_counts, bins=50, color='steelblue', edgecolor='white')
axes[0, 0].set_title('User Rating Count Distribution')
axes[0, 0].set_xlabel('Number of Ratings per User')
axes[0, 0].set_ylabel('Number of Users')

# 영화별 평점 수 분포
axes[0, 1].hist(item_rating_counts, bins=50, color='coral', edgecolor='white')
axes[0, 1].set_title('Movie Rating Count Distribution')
axes[0, 1].set_xlabel('Number of Ratings per Movie')
axes[0, 1].set_ylabel('Number of Movies')

# 평점 수 상위 20개 영화
top_movies = item_rating_counts.sort_values(ascending=False).head(20)
top_movie_titles = [movie_names.get(i, str(i))[:20] for i in top_movies.index]
axes[1, 0].barh(range(20), top_movies.values, color='coral')
axes[1, 0].set_yticks(range(20))
axes[1, 0].set_yticklabels(top_movie_titles, fontsize=8)
axes[1, 0].set_title('Top 20 Most Rated Movies')
axes[1, 0].set_xlabel('Number of Ratings')
axes[1, 0].invert_yaxis()

# 롱테일 시각화 (영화 평점 수 누적)
sorted_counts = item_rating_counts.sort_values(ascending=False)
cumulative = sorted_counts.cumsum() / sorted_counts.sum()
axes[1, 1].plot(range(len(cumulative)), cumulative.values, color='purple')
axes[1, 1].axhline(y=0.8, color='red', linestyle='--', label='80% of ratings')
axes[1, 1].set_title('Long Tail: Cumulative Rating Coverage')
axes[1, 1].set_xlabel('Movie Rank (by popularity)')
axes[1, 1].set_ylabel('Cumulative Rating Fraction')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

# 통계 요약
top_20pct = int(len(sorted_counts) * 0.2)
top_20pct_share = sorted_counts.iloc[:top_20pct].sum() / sorted_counts.sum()
print(f"상위 20% 영화({top_20pct}편)가 전체 평점의 {top_20pct_share:.1%}를 차지합니다.")
print(f"사용자 1인당 평균 평점 수: {user_rating_counts.mean():.1f}편")
print(f"영화 1편당 평균 평점 수: {item_rating_counts.mean():.1f}개")

### 🔬 코드 해설
- 평점 수 분포는 **오른쪽으로 긴 꼬리(Long Tail)** 모양입니다. 소수의 인기 영화에 평점이 몰리고, 대다수의 영화는 평점이 매우 적습니다.
- 이는 협업 필터링에서 중요한 함의를 가집니다. 평점이 적은 아이템(비인기 영화)은 유사도 계산의 신뢰성이 낮아져, 추천 시스템이 인기 아이템 중심으로 편향될 수 있습니다.
- 누적 커버리지 그래프에서 상위 20% 영화가 전체 평점의 대부분을 차지하는 것을 볼 수 있습니다.

---
# Part 2: Item-based CF 구현

`lab_01`에서는 사용자 간 유사도를 계산했습니다. 이번에는 **영화 간 유사도**를 계산해 Item-based CF를 구현합니다.
아이디어: '내가 5점을 준 영화 A와 유사한 영화 B를 추천한다.'

## 3-1. 아이템 유사도 행렬 계산
User-based CF에서는 `ratings`의 **행**을 사용자 벡터로 봤습니다. Item-based CF에서는 `ratings`의 **열**을 아이템 벡터로 봅니다. 즉, `ratings.T`를 사용합니다.

In [ ]:
# ratings: (943, 1682) — 사용자 × 영화
# ratings.T: (1682, 943) — 영화 × 사용자, 각 행이 한 영화의 평점 벡터

item_vecs = ratings.T  # (1682, 943)

item_norms = torch.norm(item_vecs, dim=1, keepdim=True).clamp(min=1e-8)  # (1682, 1)
item_normalized = item_vecs / item_norms                                   # (1682, 943)

item_sim = torch.mm(item_normalized, item_normalized.T)  # (1682, 1682)

print(f"아이템 유사도 행렬 크기: {item_sim.shape}")
print(f"대각선(자기 자신과의 유사도): {item_sim.diagonal()[:5]}")

### 🔬 코드 해설
- User-based CF와 코드 구조가 완전히 동일합니다. 달라진 것은 오직 **입력 행렬을 `ratings`에서 `ratings.T`로 바꾼 것**뿐입니다.
- `ratings.T`의 각 행은 '이 영화를 어떤 사람들이 몇 점을 줬는가'를 나타내는 벡터입니다. 이 벡터들의 코사인 유사도가 높다는 것은, 비슷한 사용자 집단이 두 영화를 비슷하게 평가했다는 의미입니다.
- 결과 행렬 `item_sim`은 1682 × 1682 크기로, `[i][j]`가 영화 i와 j의 유사도를 나타냅니다.

## 3-2. 유사 영화 검색
특정 영화와 유사도가 높은 영화들을 찾습니다.

In [ ]:
def find_similar_movies(movie_id, item_sim, movie_names, top_n=10):
    """movie_id: 1-indexed MovieLens ID"""
    item_idx = movie_id - 1
    sim_scores = item_sim[item_idx].clone()
    sim_scores[item_idx] = -1  # 자기 자신 제외

    top_indices = torch.topk(sim_scores, top_n).indices
    results = [
        (movie_names.get(idx.item() + 1, 'Unknown'), sim_scores[idx].item())
        for idx in top_indices
    ]
    return results


# 영화 ID 1번: Toy Story (1995)
query_movie_id = 1
print(f"'{movie_names[query_movie_id]}'와 유사한 영화 Top 10:\n")
for rank, (title, score) in enumerate(find_similar_movies(query_movie_id, item_sim, movie_names), 1):
    print(f"  {rank:2d}. {title:<40s} (유사도: {score:.4f})")

### 🔬 코드 해설
- `find_similar_movies`는 주어진 영화와 코사인 유사도가 높은 영화를 찾아 반환합니다.
- 유사도가 높다는 것은 '이 영화를 좋아한 사람들이 저 영화도 비슷하게 평가했다'는 의미이므로, 콘텐츠(장르, 감독 등)를 직접 분석하지 않아도 유사한 영화를 찾을 수 있습니다.

> 💡 **직접 해보기**: `query_movie_id`를 다른 영화 ID로 바꿔보세요. `movie_names` 딕셔너리에서 흥미로운 영화를 골라 유사 영화를 탐색해보세요.

## 3-3. Item-based CF 추천 생성
사용자가 높게 평가한 영화들과 유사한 영화를 추천합니다.

In [ ]:
def recommend_item_based(user_idx, ratings, item_sim, top_n=10):
    user_ratings = ratings[user_idx]        # (1682,) — 이 사용자의 평점 벡터
    unrated_mask = (user_ratings == 0)      # 아직 보지 않은 영화

    # 사용자가 평가한 각 영화에 대해 유사 영화 점수를 계산
    # item_sim: (1682, 1682)  user_ratings: (1682,)
    # item_sim * user_ratings: 각 영화(행)에 대해 사용자 평점을 가중치로 곱한 행렬
    scores = torch.mv(item_sim, user_ratings)  # (1682,) — 각 영화의 추천 점수

    scores[~unrated_mask] = -float('inf')   # 이미 본 영화 제외
    top_items = torch.topk(scores, top_n).indices
    return top_items.tolist()


user_idx = 0
rec_items = recommend_item_based(user_idx, ratings, item_sim)

print(f"사용자 {user_idx + 1}번의 Item-based CF 추천 영화 Top 10:")
for rank, item_idx in enumerate(rec_items, 1):
    print(f"  {rank}. {movie_names.get(item_idx + 1, 'Unknown')}")

### 🔬 코드 해설
- **`torch.mv(item_sim, user_ratings)`**: 행렬-벡터 곱입니다. `item_sim`의 각 행 i는 영화 i와 모든 다른 영화의 유사도입니다. 이 행과 `user_ratings`(사용자의 평점 벡터)의 내적을 계산하면, '이 영화와 사용자가 높게 평가한 영화들의 유사도 합산'이 됩니다.
- 결과적으로 `scores[i]`가 높다는 것은 '사용자가 높게 평가한 영화들과 영화 i가 유사하다'는 의미입니다. 이것이 Item-based CF의 핵심 로직입니다.

> 💡 **User-based vs Item-based 비교**: `user_idx = 0`으로 동일하게 설정하고 `lab_01`의 User-based 추천 결과와 비교해보세요. 어떤 영화들이 겹치고, 어떤 영화들이 다른가요?

## 3-4. User-based vs Item-based 추천 결과 비교

In [ ]:
# User-based CF
norms = torch.norm(ratings, dim=1, keepdim=True).clamp(min=1e-8)
normalized = ratings / norms
user_sim = torch.mm(normalized, normalized.T)

def recommend_user_based(user_idx, ratings, user_sim, top_n_neighbors=20, top_n_items=10):
    sim_scores = user_sim[user_idx].clone()
    sim_scores[user_idx] = -1
    top_neighbors = torch.topk(sim_scores, top_n_neighbors).indices
    unrated_mask = (ratings[user_idx] == 0)
    neighbor_ratings = ratings[top_neighbors]
    weights = sim_scores[top_neighbors].unsqueeze(1)
    scores = (neighbor_ratings * weights).sum(dim=0)
    scores[~unrated_mask] = -float('inf')
    return torch.topk(scores, top_n_items).indices.tolist()


user_idx = 0
ub_recs = set(recommend_user_based(user_idx, ratings, user_sim))
ib_recs = set(recommend_item_based(user_idx, ratings, item_sim))

overlap = ub_recs & ib_recs
ub_only = ub_recs - ib_recs
ib_only = ib_recs - ub_recs

print(f"사용자 {user_idx + 1}번의 두 방식 추천 결과 비교\n")
print(f"[겹치는 영화 ({len(overlap)}편)]")
for idx in overlap:
    print(f"  - {movie_names.get(idx + 1, 'Unknown')}")

print(f"\n[User-based에만 있는 영화 ({len(ub_only)}편)]")
for idx in ub_only:
    print(f"  - {movie_names.get(idx + 1, 'Unknown')}")

print(f"\n[Item-based에만 있는 영화 ({len(ib_only)}편)]")
for idx in ib_only:
    print(f"  - {movie_names.get(idx + 1, 'Unknown')}")

### 🔬 코드 해설
- 두 방식의 결과가 완전히 일치하지 않는 것은 자연스럽습니다. 추천의 관점 자체가 다르기 때문입니다.
- **User-based**: '나와 비슷한 사람들이 좋아했기 때문에 추천합니다'
- **Item-based**: '당신이 좋아했던 영화들과 비슷하기 때문에 추천합니다'
- 겹치는 영화가 있다면, 두 관점 모두에서 추천될 만큼 타당한 추천임을 시사합니다.

## 4. ✅ 학습 결과 정리

**Part 1 EDA에서:**
- 평점 분포가 3~4점에 치우친 긍정 편향(Positive Bias)을 확인했습니다.
- 93% 이상의 칸이 0인 희소(Sparse) 행렬 구조를 시각적으로 확인했습니다.
- 소수의 인기 영화가 대부분의 평점을 차지하는 롱테일(Long Tail) 현상을 발견했습니다.

**Part 2 Item-based CF에서:**
- User-based CF에서 `ratings` → Item-based CF에서 `ratings.T`로 전치만 바꾸면 코드 구조가 동일함을 확인했습니다.
- `torch.mv(item_sim, user_ratings)`로 사용자의 평점 이력과 아이템 유사도를 결합해 추천 점수를 계산했습니다.
- 두 방식의 추천 결과를 비교하며 관점의 차이를 체감했습니다.

🎯 **핵심 결론:** 데이터를 먼저 파악(EDA)해야 모델의 한계와 특성을 이해할 수 있습니다. 협업 필터링은 간단한 아이디어에서 출발하지만, 실제 데이터의 희소성과 편향이 모델 설계의 핵심 과제임을 직접 확인했습니다.